In [0]:
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType
import pyspark.sql.functions as f 
from pyspark.sql import Row

In [0]:
catalog_name = 'ecommerce'

In [0]:
df_products = spark.table(f'{catalog_name}.silver.slv_products')
df_category = spark.table(f'{catalog_name}.silver.slv_category')
df_brand = spark.table(f'{catalog_name}.silver.slv_brand')

In [0]:
df_products.createOrReplaceTempView("v_products")
df_category.createOrReplaceTempView("v_category")
df_brand.createOrReplaceTempView("v_brand")

In [0]:
spark.sql(f'USE CATALOG {catalog_name}')


In [0]:
%sql 
CREATE OR REPLACE TABLE gold.gld_dim_products AS 

WITH brands_category AS (
SELECT 
  b.brand_name,
  b.brand_code,
  c.category_name,
  c.category_code
FROM v_brand as b 
INNER JOIN v_category as c 
ON 
  b.category_code = c.category_code
)

SELECT 
  p.product_id,
  p.sku,
  bc.category_code,
  COALESCE(bc.category_name, 'Not Available') as category_name,
  bc.brand_code,
  COALESCE(bc.brand_name, 'Not Available') as brand_name,
  p.color,
  p.size,
  p.material,
  p.weight_grams,
  p.length_cm,
  p.width_cm,
  p.height_cm,
  p.rating_count,
  p.file_name,
  p.ingest_timestamp
FROM v_products as p 
LEFT JOIN brands_category as bc 
ON 
  p.brand_code =  bc.brand_code;

In [0]:
# India states
india_region = {
    "MH": "West", "GJ": "West", "RJ": "West",
    "KA": "South", "TN": "South", "TS": "South", "AP": "South", "KL": "South",
    "UP": "North", "WB": "North", "DL": "North"
}
# Australia states
australia_region = {
    "VIC": "SouthEast", "WA": "West", "NSW": "East", "QLD": "NorthEast"
}

# United Kingdom states
uk_region = {
    "ENG": "England", "WLS": "Wales", "NIR": "Northern Ireland", "SCT": "Scotland"
}

# United States states
us_region = {
    "MA": "NorthEast", "FL": "South", "NJ": "NorthEast", "CA": "West", 
    "NY": "NorthEast", "TX": "South"
}

# UAE states
uae_region = {
    "AUH": "Abu Dhabi", "DU": "Dubai", "SHJ": "Sharjah"
}

# Singapore states
singapore_region = {
    "SG": "Singapore"
}

# Canada states
canada_region = {
    "BC": "West", "AB": "West", "ON": "East", "QC": "East", "NS": "East", "IL": "Other"
}

# Combine into a master dictionary
country_state_map = {
    "India": india_region,
    "Australia": australia_region,
    "United Kingdom": uk_region,
    "United States": us_region,
    "United Arab Emirates": uae_region,
    "Singapore": singapore_region,
    "Canada": canada_region
}  
# Flatten country_state_map into a list of Rows
rows = []
for country, states in country_state_map.items():
    for state_code, region in states.items():
        rows.append(Row(country=country, state=state_code, region=region))
rows[:10]        

# 2️ Create mapping DataFrame
df_region_mapping = spark.createDataFrame(rows)


df_region_mapping.show(truncate=False)

In [0]:
df_slv_customers = spark.table(f'{catalog_name}.silver.slv_customers')
display(df_slv_customers.limit(10))


In [0]:
df_gold = df_slv_customers.join(df_region_mapping, on=["country", "state"], how="left")


In [0]:
df_gold = df_gold.fillna({
    'region': 'other'
})


In [0]:
df_gold.write.format('delta') \
    .mode('overwrite') \
        .option('mergeSchema', 'true') \
        .saveAsTable(f'{catalog_name}.gold.gld_dim_customers')

In [0]:
df_slv_calendar = spark.table(f'{catalog_name}.silver.slv_calendar')



In [0]:
df_gold = df_slv_calendar.withColumn(
    'date_id',
    f.date_format(f.col('date'), 'yyyyMMdd').cast('int')
)


In [0]:
df_gold = df_gold.withColumn(
    'month_name',
    f.date_format(f.col('date'), 'MMMM')
)

df_gold = df_gold.withColumn(
    'is_weekend',
    f.when(
        f.col('day_name').isin('Sunday', 'Saturday'),
        1
    ).otherwise(0)
)

In [0]:
desired_columns_order = ["date_id", "date", "year", "month_name", "day_name", "is_weekend", "quarter", "week", "_ingested_at", "_source_file"]

df_gold = df_gold.select(desired_columns_order)



In [0]:
df_gold.write.format('delta') \
    .mode('overwrite') \
        .option('overwriteSchema', 'true') \
        .saveAsTable(f'{catalog_name}.gold.gld_dim_calendar')